# AsyncFlow — MMc Theory vs Simulation (Guided Notebook)

This notebook shows how to:

1. Make imports work inside a notebook (src-layout or package install)
2. Build a **multi-server** scenario compatible with **M/M/c** assumptions
3. Run the simulation and collect results
4. Compare theory vs observed KPIs (pretty-printed table)
5. Plot the standard dashboards (latency, throughput, server time series)




In [19]:
import sys, importlib


for m in list(sys.modules):
    if m.startswith("asyncflow"):
        del sys.modules[m]


from asyncflow import AsyncFlow, SimulationRunner
from asyncflow.analysis import MMc, ResultsAnalyzer
from asyncflow.components import (
    Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
)
from asyncflow.settings import SimulationSettings

import simpy

In [20]:
import matplotlib.pyplot as plt
import simpy

# Public AsyncFlow API
from asyncflow import AsyncFlow, SimulationRunner, Sweep
from asyncflow.components import Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
from asyncflow.settings import SimulationSettings
from asyncflow.analysis import  ResultsAnalyzer, SweepAnalyzer, MMc
from asyncflow.enums import Distribution

print("Imports OK.")

Imports OK.


## 1) Build an M/M/c split-friendly scenario

* **Multiple identical servers with exponential CPU service**
  Topology includes **\$c \geq 2\$ identical servers**, each exposing exactly **one endpoint** with exactly **one CPU-bound step**.
  Service times follow an **Exponential** distribution with mean \$E\[S]\$ (service rate \$\mu = 1/E\[S]\$). No RAM/IO steps are included in the pipeline.

* **Load balancer with FCFS dispatch**

* **“Poisson arrivals” via the generator**
  
  

---

```mermaid
graph LR;
    rqs1["<b>RqsGenerator</b><br/>id: rqs-1"]
    client1["<b>Client</b><br/>id: client-1"]
    lb1["<b>LoadBalancer</b><br/>id: lb-1<br/>Policy: round_robin"]
    app1["<b>Server</b><br/>id: app-1<br/>Endpoint: /api"]
    app2["<b>Server</b><br/>id: app-2<br/>Endpoint: /api"]

    rqs1 -- "Edge: gen-client<br/>Latency: 0.0001" --> client1;
    client1 -- "Request<br/>Edge: client-lb<br/>Latency: 0.0001" --> lb1;
    lb1 -- "Dispatch<br/>Edge: lb-app1<br/>Latency: 0.0001" --> app1;
    lb1 -- "Dispatch<br/>Edge: lb-app2<br/>Latency: 0.0001" --> app2;
    app1 -- "Response<br/>Edge: app1-client<br/>Latency: 0.0001" --> client1;
    app2 -- "Response<br/>Edge: app2-client<br/>Latency: 0.0001" --> client1;
```

---



In [21]:
def build_payload():
    generator = ArrivalsGenerator(
        id="rqs-1",
        lambda_rps=270,
        model=Distribution.POISSON
    )

    client = Client(id="client-1")

    endpoint = Endpoint(
        endpoint_name="/api",
        probability=1.0,
        steps=[
            {
                "kind": "initial_parsing",
                "step_operation": {
                    "cpu_time": {"mean": 0.01, "distribution": "exponential"},
                },
            },
        ],
    )

    srv1 = Server(
        id="srv-1",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    srv2 = Server(
        id="srv-2",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    
    srv3 = Server(
        id="srv-3",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )

    lb = LoadBalancer(
        id="lb-1",
        algorithms="fcfs",  
        server_covered={"srv-1", "srv-2", "srv-3"},
    )

    edges = [
        LinkEdge(id="gen-client",  source="rqs-1",  target="client-1",),
        LinkEdge(id="client-lb",   source="client-1", target="lb-1",  ),
        LinkEdge(id="lb-srv1",     source="lb-1",   target="srv-1",   ),
        LinkEdge(id="lb-srv2",     source="lb-1",   target="srv-2",   ),
        LinkEdge(id="lb-srv3",     source="lb-1",   target="srv-3",   ),
        LinkEdge(id="srv1-client", source="srv-1",  target="rqs-1",),
        LinkEdge(id="srv2-client", source="srv-2",  target="rqs-1",),
        LinkEdge(id="srv3-client", source="srv-3",  target="rqs-1",),
    ]

    settings = SimulationSettings(
        total_simulation_time=3600,
        sample_period_s=0.05,
    )

    payload = (
        AsyncFlow()
        .add_arrivals_generator(generator)
        .add_client(client)
        .add_servers(srv1, srv2, srv3)
        .add_load_balancer(lb)
        .add_edges(*edges)
        .add_simulation_settings(settings)
    ).build_payload()

    return payload


## 2) Run the simulation

In [22]:
payload = build_payload()
env = simpy.Environment()
runner = SimulationRunner(env=env, simulation_input=payload)
results: ResultsAnalyzer = runner.run()
print("Done.")


Done.


# 3) M/M/c (FCFS) — theory vs observed comparison

This section shows how we compute the **theoretical Erlang-C KPIs** (pooled queue, FCFS) and compare them against **simulation estimates**.

---

## Variables

* **$c$**: number of identical servers.
* **$\lambda$**: global arrival rate (req/s).
* **$\mu$**: per-server service rate (req/s), $\mu = 1/\mathbb{E}[S]$.
* **$\rho$**: global utilization, $\rho = \lambda/(c\mu)$.
* **$W$**: mean time in system (queue + service).
* **$W_q$**: mean waiting time in queue.
* **$L$**: mean number in system.
* **$L_q$**: mean number in queue.

---

## Theory (Erlang-C formulas)

We assume **Poisson arrivals** for $\lambda$ (taken directly from the payload).

1. Offered load:

$$
a = \frac{\lambda}{\mu}
$$

2. Probability system is empty:

$$
P_0 = \left[\sum_{n=0}^{c-1}\frac{a^n}{n!} + \frac{a^c}{c!\,(1-\rho)}\right]^{-1}
$$

3. Probability of waiting (Erlang-C):

$$
P_w = \frac{a^c}{c!\,(1-\rho)} \, P_0
$$

4. Queue length and waiting:

$$
L_q = P_w \cdot \frac{\rho}{1-\rho}, \qquad
W_q = \frac{L_q}{\lambda}
$$

5. Total response time and system size:

$$
W = W_q + \frac{1}{\mu}, \qquad
L = \lambda W
$$

If $\rho \ge 1$, the system is unstable and all metrics diverge to $+\infty$.

---

## Observed (from simulation, FCFS/Erlang-C)

Under FCFS we derive **each KPI directly** from sampled time series or per-request buckets, **not** by deriving one observed KPI from another. This keeps Little’s law checks as **independent validations**, not definitions.

1. **Arrival rate**

$$
\lambda_{\text{obs}}=\mathrm{mean}(\text{Throughput RPS})
$$

Mean of the throughput series (client completions per fixed window, typically 1 s).

2. **Service rate**

$$
\mu_{\text{obs}}=\frac{1}{\overline{S}},\quad 
\overline{S}=\mathrm{mean}(\texttt{SERVICE\_TIME})
$$

Computed from per-request server buckets, aggregated across all servers.

3. **End-to-end time**

$$
W_{\text{obs}}=\mathrm{mean}(\text{client latencies})
$$

From the generator’s request clocks (start/finish per request).

4. **Queue waiting time (LB)**

$$
W_{q,\text{obs}}=\mathrm{mean}(\text{LB waiting times})
$$

Mean of the **load balancer FCFS** waiting time recorded per request (not inferred from queue length).

5. **Mean number in system**

$$
L_{\text{obs}}=\mathrm{mean}(\texttt{L\_SYSTEM})
$$

Time average of the sampled series of **concurrent requests in the system** (sampled every $\Delta t$).
*Fallback if the series is unavailable in tests:* $L_{\text{obs}}=\lambda_{\text{obs}} W_{\text{obs}}$.

6. **Mean number in queue (LB)**

$$
L_{q,\text{obs}}=\mathrm{mean}(\texttt{LQ\_LB})
$$

Time average of the **LB queue length** series.
*Fallback:* $L_{q,\text{obs}}=\lambda_{\text{obs}} W_{q,\text{obs}}$.

7. **Global utilization**

$$
\rho_{\text{obs}}=\mathrm{mean}(\texttt{SERVER\_UTILIZATION})
$$

Each server samples a 0/1 “busy” indicator; $\rho_{\text{obs}}$ is the time average (with 1 core/server it matches the busy fraction).
*Fallback:* $\rho_{\text{obs}}=\lambda_{\text{obs}}/(c\,\mu_{\text{obs}})$.

> Because each quantity comes from its own primary measurement (series or buckets), cross-relations like $L=\lambda W$ and $L_q=\lambda W_q$ are genuine **consistency checks**, not tautologies.


---

## Comparison

The analyzer builds a table with two columns — **Theory** (Erlang-C closed forms) and **Observed** (empirical estimates) — and reports absolute and relative deltas.

This allows us to verify whether AsyncFlow reproduces the textbook M/M/c (FCFS) predictions under Poisson arrivals and exponential service.




In [23]:
mmc = MMc()
if mmc.is_compatible(payload):
   mmc.print_comparison(payload, results)  
else:
    print("Payload is not compatible with M/M/c:")
    for reason in mmc.explain_incompatibilities(payload):
        print(" -", reason)
   


MMc (FCFS/Erlang-C) — Theory vs Observed
-------------------------------------------------------------------
sym  metric                    theory    observed        abs   rel%
-------------------------------------------------------------------
λ    Arrival rate (1/s)    270.000000  269.751667  -0.248333  -0.09
μ    Service rate (1/s)    100.000000  100.058789   0.058789   0.06
rho  Utilization             0.900000    0.898517  -0.001483  -0.16
L    Mean items in sys      10.053549    9.865831  -0.187718  -1.87
Lq   Mean items in queue     7.353549    7.170280  -0.183269  -2.49
W    Mean time in sys (s)    0.037235    0.036556  -0.000680  -1.83
Wq   Mean waiting (s)        0.027235    0.026562  -0.000674  -2.47
